# MediaPipe pose + hands overlay videos

Uses the **MediaPipe Tasks** API (**Pose landmarker** + **Hand landmarker**) to estimate **body pose** and **both hands**, draws landmarks on each frame, and saves new MP4s under **`Videos/`** (originals unchanged).

> **Why not `mediapipe.solutions`?** Pip releases **≥ 0.10.31** removed the legacy **Solutions** API (`Holistic`), which causes `AttributeError: module 'mediapipe' has no attribute 'solutions'`. This notebook uses **Tasks**, which matches current packages.

**Input / output**

- Reads **`Videos/ASL_60`** if it exists; otherwise **`Videos/ASL`**.
- Reads **`Videos/SGSL`** if present.
- Writes **overlay MP4s** under **`Videos/ASL_60_pose_hands_video/...`** (from **`ASL_60`**) or **`Videos/ASL_pose_hands_video/...`** if only **`ASL`** exists; **`Videos/SGSL_pose_hands_video/...`** for **SGSL**.
- Writes **keypoints** under parallel trees **`..._keypoints/`** (same **`/<class>/…`** layout).
- Per clip: **`/<class>/<n>.mp4`** in the **`_video`** tree; **`frame_XXXXXX.npy`** (shape **`(75, 5)`**) in **`/<class>/<n>/`** under the **`_keypoints`** tree (cols: **x, y, z, visibility, presence**; pose face rows **0–10** → **NaN**; hands **left 33–53**, **right 54–74**).

**Encoding:** MP4 is written with **`ffmpeg` + libx264** (piped BGR frames). OpenCV’s **`mp4v`** writer often fails with odd FPS (**MPEG‑4 timebase** limit → install **`ffmpeg`** on PATH, e.g. `brew install ffmpeg` on macOS).

**Models:** On first run, **`.lite`** pose + **hand_landmarker** `.task` bundles are downloaded into **`.cache/mediapipe_models/`** beside the notebook (internet required once).

```bash
pip install "mediapipe>=0.10.14" "opencv-python-headless>=4.8" "numpy>=1.26,<2"
```

```bash
# macOS — encode MP4 (libx264) from piped frames
brew install ffmpeg
```

Keeping **NumPy < 2** avoids common resolver noise with scipy / langchain in shared Jupyter envs (`01_*`, `02_*` use the same pin). Restart the kernel after `%pip`.



In [1]:
%pip install -q "mediapipe>=0.10.14" "opencv-python-headless>=4.8" "numpy>=1.26,<2"



[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
from __future__ import annotations

from pathlib import Path

import cv2
import numpy as np

PROJECT_ROOT = Path(".").resolve()

ASL_IN = PROJECT_ROOT / "Videos" / "ASL_60"
_used_asl_60 = True
if not ASL_IN.is_dir():
    ASL_IN = PROJECT_ROOT / "Videos" / "ASL"
    _used_asl_60 = False

SGSL_IN = PROJECT_ROOT / "Videos" / "SGSL"

# Outputs: videos and per-frame keypoints live in **separate** folder trees.
ASL_VIDEOS_OUT = (
    PROJECT_ROOT / "Videos" / "ASL_60_pose_hands_video"
    if _used_asl_60
    else PROJECT_ROOT / "Videos" / "ASL_pose_hands_video"
)
ASL_KEYPOINTS_OUT = (
    PROJECT_ROOT / "Videos" / "ASL_60_pose_hands_keypoints"
    if _used_asl_60
    else PROJECT_ROOT / "Videos" / "ASL_pose_hands_keypoints"
)

PROCESS_JOBS: list[tuple[str, Path, Path, Path]] = [
    ("ASL", ASL_IN, ASL_VIDEOS_OUT, ASL_KEYPOINTS_OUT),
    (
        "SGSL",
        SGSL_IN,
        PROJECT_ROOT / "Videos" / "SGSL_pose_hands_video",
        PROJECT_ROOT / "Videos" / "SGSL_pose_hands_keypoints",
    ),
]

# Shared detection / tracking thresholds for pose + hands Tasks models
MIN_DET_CONF = 0.5
MIN_TRK_CONF = 0.5
NUM_HANDS = 2

# Save one `.npy` array per decoded frame under the keypoints root (see markdown).
SAVE_KEYPOINTS = True



In [3]:
import shutil
import subprocess
import dataclasses
import urllib.request

from mediapipe.tasks.python.core import base_options as mp_base_opts
from mediapipe.tasks.python.vision import drawing_styles as mp_styles
from mediapipe.tasks.python.vision import drawing_utils as mp_draw
from mediapipe.tasks.python.vision.hand_landmarker import (
    HandLandmarker,
    HandLandmarkerOptions,
    HandLandmarkerResult,
    HandLandmarksConnections,
)
from mediapipe.tasks.python.vision.pose_landmarker import (
    PoseLandmarker,
    PoseLandmarkerOptions,
    PoseLandmarkerResult,
    PoseLandmarksConnections,
)
from mediapipe.tasks.python.vision.core import image as mp_image_lib
from mediapipe.tasks.python.vision.core.image import ImageFormat
from mediapipe.tasks.python.vision.core import vision_task_running_mode as vtrm

_RUNNING = vtrm.VisionTaskRunningMode.VIDEO

# Official Google-hosted .task bundles
MODEL_POSE_LITE = (
    "https://storage.googleapis.com/mediapipe-models/pose_landmarker/"
    "pose_landmarker_lite/float16/1/pose_landmarker_lite.task"
)
MODEL_HAND = (
    "https://storage.googleapis.com/mediapipe-models/hand_landmarker/"
    "hand_landmarker/float16/1/hand_landmarker.task"
)


def _ensure_model(url: str, local_name: str) -> Path:
    d = PROJECT_ROOT / ".cache" / "mediapipe_models"
    d.mkdir(parents=True, exist_ok=True)
    pth = d / local_name
    if pth.is_file() and pth.stat().st_size > 0:
        return pth
    print(f"Downloading Mediapipe model → {pth.name} …")
    urllib.request.urlretrieve(url, str(pth))
    return pth


def _open_h264_mp4_ffmpeg_pipe(path: Path, w: int, h: int, fps: float) -> subprocess.Popen:
    """Stream BGR frames to H.264 MP4 (matches `02_*` notebook — avoids mp4v timebase failures)."""
    path.parent.mkdir(parents=True, exist_ok=True)
    fps_cmd = float(fps)
    if fps_cmd <= 1e-6:
        fps_cmd = 30.0

    ff = shutil.which("ffmpeg") or "ffmpeg"
    proc = subprocess.Popen(
        [
            ff,
            "-y",
            "-f",
            "rawvideo",
            "-vcodec",
            "rawvideo",
            "-pix_fmt",
            "bgr24",
            "-s",
            f"{w}x{h}",
            "-r",
            str(fps_cmd),
            "-i",
            "-",
            "-an",
            "-c:v",
            "libx264",
            "-pix_fmt",
            "yuv420p",
            "-preset",
            "veryfast",
            "-crf",
            "20",
            "-movflags",
            "+faststart",
            str(path),
        ],
        stdin=subprocess.PIPE,
        stdout=subprocess.DEVNULL,
        stderr=subprocess.PIPE,
    )
    if proc.stdin is None:
        raise RuntimeError("ffmpeg stdin unavailable")
    return proc


def _finalize_ffmpeg(proc: subprocess.Popen, path: Path) -> None:
    try:
        proc.stdin.close()
    except BrokenPipeError:
        pass
    err = ""
    if proc.stderr:
        err = proc.stderr.read().decode("utf-8", errors="replace")
    if proc.wait() != 0:
        raise RuntimeError(
            f"ffmpeg failed for {path}. Install ffmpeg on PATH (e.g. brew install ffmpeg).\n"
            "--- stderr (tail) ---\n"
            + err[-4000:]
        )


# Pose indices 0–10: nose, eyes, ears, mouth — omit from overlay (not informative for sign language).
_POSE_FACE_IDX = frozenset(range(11))

_POSE_BODY_ONLY_CONNECTIONS = [
    c
    for c in PoseLandmarksConnections.POSE_LANDMARKS
    if c.start not in _POSE_FACE_IDX and c.end not in _POSE_FACE_IDX
]


def _pose_landmarks_hide_face(landmarks: list) -> list:
    """Drop face keypoints from pose drawing (visibility gate in drawing_utils)."""
    return [
        dataclasses.replace(lm, visibility=0.0, presence=0.0)
        if i in _POSE_FACE_IDX
        else lm
        for i, lm in enumerate(landmarks)
    ]


def list_numeric_mp4s(folder: Path) -> list[Path]:
    return sorted(folder.glob("*.mp4"), key=lambda p: int(p.stem))


def draw_overlay_tasks(
    frame_bgr: np.ndarray,
    pose_result: PoseLandmarkerResult,
    hand_result: HandLandmarkerResult,
) -> np.ndarray:
    out = frame_bgr.copy()

    pose_point_style = mp_styles.get_default_pose_landmarks_style()
    pose_conn_style = mp_draw.DrawingSpec(color=(224, 224, 224), thickness=3)

    if pose_result.pose_landmarks:
        plm = _pose_landmarks_hide_face(pose_result.pose_landmarks[0])
        mp_draw.draw_landmarks(
            out,
            plm,
            _POSE_BODY_ONLY_CONNECTIONS,
            landmark_drawing_spec=pose_point_style,
            connection_drawing_spec=pose_conn_style,
        )

    hand_pt = mp_styles.get_default_hand_landmarks_style()
    hand_cn = mp_styles.get_default_hand_connections_style()
    if hand_result.hand_landmarks:
        for hlm in hand_result.hand_landmarks:
            mp_draw.draw_landmarks(
                out,
                hlm,
                HandLandmarksConnections.HAND_CONNECTIONS,
                landmark_drawing_spec=hand_pt,
                connection_drawing_spec=hand_cn,
            )
    return out


_N_POSE = 33
_N_HAND = 21
_KEYPOINT_ROWS = _N_POSE + 2 * _N_HAND  # pose + LH + RH
_KEYPOINT_FEATURES = 5  # x, y, z, visibility, presence (normalized image space)


def _normalized_lm_to_vec(lm) -> np.ndarray:
    def g(attr: str) -> float:
        v = getattr(lm, attr)
        return float(v) if v is not None else np.nan

    return np.array(
        [
            g("x"),
            g("y"),
            g("z"),
            g("visibility"),
            g("presence"),
        ],
        dtype=np.float32,
    )


def pack_frame_keypoints(
    pose_result: PoseLandmarkerResult,
    hand_result: HandLandmarkerResult,
) -> np.ndarray:
    """`(75, 5)` pose (33; face rows absent → NaN) + left/right hand landmarks (21 each)."""
    out = np.full((_KEYPOINT_ROWS, _KEYPOINT_FEATURES), np.nan, dtype=np.float32)

    if pose_result.pose_landmarks:
        pl = pose_result.pose_landmarks[0]
        for i in range(min(_N_POSE, len(pl))):
            if i in _POSE_FACE_IDX:
                continue
            out[i] = _normalized_lm_to_vec(pl[i])

    if not hand_result.hand_landmarks:
        return out

    left_sl = slice(_N_POSE, _N_POSE + _N_HAND)
    right_sl = slice(_N_POSE + _N_HAND, _N_POSE + 2 * _N_HAND)
    ambiguous: list = []

    for handedness_cat, hlms in zip(hand_result.handedness, hand_result.hand_landmarks):
        label = ""
        if handedness_cat:
            cat0 = handedness_cat[0]
            label = (cat0.category_name or cat0.display_name or "") or ""
        low = label.lower()
        if "left" in low:
            tgt = left_sl
        elif "right" in low:
            tgt = right_sl
        else:
            ambiguous.append(hlms)
            continue

        for j in range(min(_N_HAND, len(hlms))):
            out[tgt.start + j] = _normalized_lm_to_vec(hlms[j])

    vacant: list[slice] = []
    if np.all(np.isnan(out[left_sl])):
        vacant.append(left_sl)
    if np.all(np.isnan(out[right_sl])):
        vacant.append(right_sl)
    for hlms, tgt in zip(ambiguous, vacant):
        for j in range(min(_N_HAND, len(hlms))):
            out[tgt.start + j] = _normalized_lm_to_vec(hlms[j])

    return out



def process_video_pose_hands(src: Path, dst_video: Path, kp_dir: Path, pose_path: Path, hand_path: Path) -> None:
    cap = cv2.VideoCapture(str(src))
    if not cap.isOpened():
        raise RuntimeError(f"Cannot open: {src}")

    fps = float(cap.get(cv2.CAP_PROP_FPS))
    if fps <= 1e-3:
        fps = 30.0

    dt_ms = max(1, int(round(1000.0 / fps)))

    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    kp_root = kp_dir
    frame_ix = 0

    proc = None
    pose_lm = None
    hand_lm = None

    try:
        if SAVE_KEYPOINTS:
            if kp_root.exists():
                shutil.rmtree(kp_root)
            kp_root.mkdir(parents=True, exist_ok=True)

        proc = _open_h264_mp4_ffmpeg_pipe(dst_video, w, h, fps)

        pose_lm = PoseLandmarker.create_from_options(
            PoseLandmarkerOptions(
                base_options=mp_base_opts.BaseOptions(model_asset_path=str(pose_path)),
                running_mode=_RUNNING,
                num_poses=1,
                min_pose_detection_confidence=MIN_DET_CONF,
                min_pose_presence_confidence=MIN_DET_CONF,
                min_tracking_confidence=MIN_TRK_CONF,
            )
        )
        hand_lm = HandLandmarker.create_from_options(
            HandLandmarkerOptions(
                base_options=mp_base_opts.BaseOptions(model_asset_path=str(hand_path)),
                running_mode=_RUNNING,
                num_hands=NUM_HANDS,
                min_hand_detection_confidence=MIN_DET_CONF,
                min_hand_presence_confidence=MIN_DET_CONF,
                min_tracking_confidence=MIN_TRK_CONF,
            )
        )

        ts_ms = 0
        stdin = proc.stdin

        while True:
            ok, frame = cap.read()
            if not ok or frame is None:
                break

            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            rgb = np.ascontiguousarray(rgb)
            mp_image = mp_image_lib.Image(ImageFormat.SRGB, rgb)

            pose_res = pose_lm.detect_for_video(mp_image, ts_ms)
            hand_res = hand_lm.detect_for_video(mp_image, ts_ms)

            if SAVE_KEYPOINTS:
                np.save(
                    kp_root / f"frame_{frame_ix:06d}.npy",
                    pack_frame_keypoints(pose_res, hand_res),
                )
                frame_ix += 1

            overlay = draw_overlay_tasks(frame, pose_res, hand_res)

            if overlay.shape[1] != w or overlay.shape[0] != h:
                overlay = cv2.resize(
                    overlay, (w, h), interpolation=cv2.INTER_LINEAR
                )

            if overlay.dtype != np.uint8:
                overlay = overlay.astype(np.uint8)
            if not overlay.flags["C_CONTIGUOUS"]:
                overlay = np.ascontiguousarray(overlay)

            stdin.write(overlay.tobytes())

            ts_ms += dt_ms

    finally:
        if pose_lm is not None:
            pose_lm.close()
        if hand_lm is not None:
            hand_lm.close()
        cap.release()
        if proc is not None:
            _finalize_ffmpeg(proc, dst_video)


def run_jobs(jobs: list[tuple[str, Path, Path, Path]]) -> None:
    pose_p = _ensure_model(MODEL_POSE_LITE, "pose_landmarker_lite.task")
    hand_p = _ensure_model(MODEL_HAND, "hand_landmarker.task")

    for label, in_root, video_root, keypoints_root in jobs:
        if not in_root.is_dir():
            try:
                rel = in_root.relative_to(PROJECT_ROOT)
            except ValueError:
                rel = in_root
            print(f"Skip {label}: not found — {rel}")
            continue

        class_dirs = sorted(
            [d for d in in_root.iterdir() if d.is_dir() and not d.name.startswith(".")],
            key=lambda d: d.name.lower(),
        )
        total = 0
        for cls in class_dirs:
            for vid in list_numeric_mp4s(cls):
                dst_video = video_root / cls.name / vid.name
                kp_clip = keypoints_root / cls.name / vid.stem
                process_video_pose_hands(vid, dst_video, kp_clip, pose_p, hand_p)
                total += 1
        try:
            vrel = video_root.relative_to(PROJECT_ROOT)
        except ValueError:
            vrel = video_root
        try:
            krel = keypoints_root.relative_to(PROJECT_ROOT)
        except ValueError:
            krel = keypoints_root
        print(
            f"{label}: {total} clips → video: {vrel}"
            + (f" | keypoints: {krel}" if SAVE_KEYPOINTS else "")
        )





In [4]:
run_jobs(PROCESS_JOBS)


I0000 00:00:1779003278.698828 12892336 init-domain.cc:128] Fiber init: default domain = pthread, concurrency = 11, prefix = pthread-default
I0000 00:00:1779003278.788495 12892336 gl_context.cc:407] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M1 Max
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1779003278.863104 12892339 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1779003278.870894 12892339 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
I0000 00:00:1779003278.877425 12892352 gl_context.cc:407] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M1 Max
W0000 00:00:1779003278.881696 12892355 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:177900327

ASL: 180 clips → video: Videos/ASL_60_pose_hands_video | keypoints: Videos/ASL_60_pose_hands_keypoints


I0000 00:00:1779003744.376184 12922249 gl_context.cc:407] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M1 Max
W0000 00:00:1779003744.453262 12922253 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1779003744.461619 12922253 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
I0000 00:00:1779003744.466150 12922264 gl_context.cc:407] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M1 Max
W0000 00:00:1779003744.471010 12922267 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1779003744.475097 12922269 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
I0000 00:00:1779003747.233575 1292

SGSL: 180 clips → video: Videos/SGSL_pose_hands_video | keypoints: Videos/SGSL_pose_hands_keypoints


### Notes

- **MediaPipe Tasks** replaces legacy **`mp.solutions` / Holistic** in current wheels; pose + hands are run as **two Tasks** on each frame (**lite** pose model keeps CPU use reasonable — swap URLs in code for heavier models).
- **First run** downloads **`.task`** files into **`.cache/mediapipe_models/`** (needs network once).
- **Videos** and **keypoint `.npy` trees** are separate folder roots (`*_pose_hands_video` vs `*_pose_hands_keypoints`).
- Outputs are encoded with **`ffmpeg` + libx264** (same rationale as **`02_*`**) — **`brew install ffmpeg`** on macOS if encoding fails.
